Stage 1: Setup and Initial Inspection


In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv')

In [5]:
print(df.shape)
print(df.dtypes)

#missing values per column
print(df.isnull().sum())

#percentage of missing values per column
missing_percentage = df.isnull().mean()
columns_to_drop = missing_percentage[missing_percentage > 0.20].index.tolist()
print(columns_to_drop)

#drop the column if has more than 20% missing values
df_cleaned = df.drop(columns=columns_to_drop)

(32833, 23)
track_id                     object
track_name                   object
track_artist                 object
track_popularity              int64
track_album_id               object
track_album_name             object
track_album_release_date     object
playlist_name                object
playlist_id                  object
playlist_genre               object
playlist_subgenre            object
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64
dtype: object
track_id                    0
track_name                  5
track_artist                5
track_popularity            0
track_album_id            

During the first and second step of Stage 1, columns with more than $20\%$ missing values were permanently removed from the active training dataset. In machine learning and predictive modeling, attempting to impute (fill in) missing values for a feature that lacks nearly a quarter of its data introduces a severe risk of data fabrication, artificial bias, and noise.Standard imputation techniques (such as substituting the mean, median, or most frequent value) require a solid underlying distribution to be accurate. When the missingness is high, imputation heavily distorts the feature's variance and weakens the model’s predictive reliability. Therefore, eliminating these sparse features prevents the recommendation model from learning flawed or artificial patterns, ensuring higher overall data integrity

In [6]:
#cek duplikat menggunakan kolom track_id
print(df.duplicated(subset=['track_id']).sum())

#deduplicate dan simpan baris pertama yang muncul
df_deduplicate = df.drop_duplicates(subset=['track_id'], keep='first')

#cek ukuran data setelah deduplicate
print(df_deduplicate.shape)

4477
(28356, 23)


Column used for deduplication: track_id
Why: A single unique song (identified by a unique track_id) can naturally appear in multiple different playlists. If we do not handle this, the machine learning model will over-represent (give too much weight to) popular songs that appear frequently across playlists.

By dropping duplicates based on the track_id column and keeping only the first occurrence (keep='first'), we ensure that each unique song’s audio features (such as danceability, energy, and tempo) are represented exactly once. This prevents data leakage and structural bias, leading to a fairer and more robust recommendation system.

In [9]:
import numpy as np

#filter kolom numerik (int dan float)
numeric_cols = df_deduplicate.select_dtypes(include=[np.number])

#statistik dasar
means = numeric_cols.mean()
medians = numeric_cols.median()
stds = numeric_cols.std()

#IQR (Q3(percentile 75) - Q1(percentile 25))
q3 = np.percentile(numeric_cols, 75, axis=0)
q1 = np.percentile(numeric_cols, 25, axis=0)
iqr = q3 - q1

summary_table = pd.DataFrame({
    'Mean': means,
    'Median': medians,
    'Standard Deviation': stds,
    'IQR': iqr
})

display(summary_table)

,Mean,Median,Standard Deviation,IQR
track_popularity,39.329771,42.000000,23.702376,37.000000
danceability,0.653372,0.670000,0.145785,0.199000
energy,0.698388,0.722000,0.183503,0.264000
key,5.368000,6.000000,3.613904,7.000000
loudness,-6.817696,-6.261000,3.036243,3.600250
mode,0.565489,1.000000,0.495701,1.000000
speechiness,0.107954,0.062600,0.102556,0.092000
acousticness,0.177176,0.079700,0.222803,0.245625
instrumentalness,0.091117,0.000021,0.232548,0.006570
liveness,0.190958,0.127000,0.155894,0.156400


Stage 2: Genre and Popularity Analysis

In [10]:
features = ['track_popularity', 'danceability', 'energy', 'valence']
genre_stats = df_deduplicate.groupby('playlist_genre')[features].agg(['mean', 'std', 'median'])
display(genre_stats)

track_popularity                   danceability            \
                           mean        std median         mean       std   
playlist_genre                                                             
edm                   30.678286  20.346961   33.0     0.657639  0.123722   
latin                 41.439691  23.394529   45.0     0.711012  0.117222   
pop                   45.905300  24.616386   50.0     0.637698  0.128997   
r&b                   35.929396  23.662984   38.0     0.667475  0.137610   
rap                   41.822811  22.765566   46.0     0.715991  0.136217   
rock                  39.694309  24.229616   44.0     0.518519  0.140418   

                         energy                    valence                   
               median      mean       std median      mean       std median  
playlist_genre                                                               
edm             0.660  0.809604  0.136550  0.838  0.397491  0.228631  0.365  
latin           0.727  0.710468  0.156036  0.733  0.607390  0.225631  0.632  
pop             0.650  0.701031  0.172949  0.727  0.502176  0.221924  0.499  
r&b             0.687  0.588932  0.181949  0.594  0.537936  0.225879  0.548  
rap             0.734  0.649832  0.172107  0.666  0.505182  0.225269  0.517  
rock            0.522  0.733067  0.197484  0.779  0.532560  0.230204  0.526

In [11]:
popularity_variance = df_deduplicate.groupby('playlist_genre')['track_popularity'].std() ** 2
print(popularity_variance.sort_values(ascending=False))

playlist_genre
pop      605.966474
rock     587.074282
r&b      559.936831
latin    547.303966
rap      518.271006
edm      413.998817
Name: track_popularity, dtype: float64


The genre with the highest variance in track popularity is pop.

Business perspective:
High variance means that the popularity of songs within this genre is highly spread out; it contains both viral mega-hits (extremely high popularity) and completely obscure tracks (very low popularity).

From a content recommendation business perspective, this indicates that a generic "popular choice" recommendation strategy will fail for this genre. The system cannot simply recommend random tracks from this genre assuming the user will like them. Instead, the recommendation engine must heavily rely on personalized features (like user history or acoustic similarity) to filter out the low-performing noise and surface the high-quality tracks, minimizing the risk of bad user experiences.

In [12]:
#10 artists with the most tracks
top_10_artists = df_deduplicate['track_artist'].value_counts().head(10).index.tolist()

#highest mean track_popularity?
top_artists_pop = df_deduplicate[df_deduplicate['track_artist'].isin(top_10_artists)] \
                    .groupby('track_artist')['track_popularity'] \
                    .agg(['count', 'mean']) \
                    .sort_values(by='mean', ascending=False)

display(top_artists_pop)

#volume and quality correlation
artist_counts = df_deduplicate['track_artist'].value_counts()
artist_mean_pop = df_deduplicate.groupby('track_artist')['track_popularity'].mean()
correlation = artist_counts.corr(artist_mean_pop)

print(correlation)


,count,mean
track_artist,,
David Guetta,81,49.370370
The Chainsmokers,66,49.227273
Queen,130,42.400000
Logic,65,41.907692
Drake,68,41.808824
Martin Garrix,87,41.402299
Don Omar,84,39.059524
Hardwell,68,36.323529
Dimitri Vegas & Like Mike,68,36.220588


0.0701243961628298


Based on the computed correlation coefficient (which is close to 0), volume does not correlate with quality in this dataset. Having a massive catalog of tracks (high volume) does not guarantee that an artist's songs will have higher average popularity (quality). For a recommendation system, this implies that the model should prioritize actual track engagement metrics rather than biasing recommendations toward prolific artists who simply release a large volume of songs.

In [13]:
condition = (
    (df_deduplicate['track_popularity'] > 70) &
    (df_deduplicate['danceability'] > 0.7) &
    (df_deduplicate['energy'] > 0.6) &
    (df_deduplicate['duration_ms'] < 240000)
)

filtered_df = df_deduplicate[condition]

#track that pass
print(len(filtered_df))

#dominating genre
print(filtered_df['playlist_genre'].value_counts())

579
playlist_genre
pop      193
latin    173
rap      141
r&b       42
edm       19
rock      11
Name: count, dtype: int64


Stage 3: NumPy Analysis

In [14]:
audio_features = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

#convert to array
X = df_deduplicate[audio_features].to_numpy()

#Min-Max Scaling ((X - min) / (max - min))
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_scaled = (X - X_min) / (X_max - X_min)

In [15]:
#np.corrcoef
corr_matrix = np.corrcoef(X_scaled.T)

#copy matriks
corr_copy = corr_matrix.copy()
np.fill_diagonal(corr_copy, np.nan)

#find highest and lowest correleation
max_idx = np.unravel_index(np.nanargmax(corr_copy), corr_copy.shape)
min_idx = np.unravel_index(np.nanargmin(corr_copy), corr_copy.shape)

highest_pos_pair = (audio_features[max_idx[0]], audio_features[max_idx[1]])
highest_pos_val = corr_matrix[max_idx]

most_neg_pair = (audio_features[min_idx[0]], audio_features[min_idx[1]])
most_neg_val = corr_matrix[min_idx]

print(f"Highest Positive Correlation: {highest_pos_pair} with value {highest_pos_val}")
print(f"Most Negative Correlation: {most_neg_pair} with value {most_neg_val}")

Highest Positive Correlation: ('energy', 'loudness') with value 0.6821377424910238
Most Negative Correlation: ('energy', 'acousticness') with value -0.5458861907892343


Musical Interpretation:

Highest Positive Correlation (energy and loudness): In musical terms, this strong positive relationship is highly intuitive. Songs that are mixed and mastered to be loud physically compress more acoustic energy. Louder tracks feel more intense, fast-paced, and dynamic, which directly drives the algorithmic perception of high "energy." You can expect this in genres like EDM, Pop, or Rock.

Most Negative Correlation (typically energy and acousticness): Musically, acoustic instruments (like acoustic guitars, pianos, or orchestral ensembles) rely on natural dynamic ranges and open space, making them inherently quieter and more mellow. As a song introduces more acoustic elements, its synthetic drive and high mechanical energy decrease. Therefore, high acousticness naturally means lower energy (e.g., classical, folk, or ambient music).

In [16]:
#convert energy and track_popularity column to array
energy_arr = df_deduplicate['energy'].to_numpy()
popularity_arr = df_deduplicate['track_popularity'].to_numpy()

#count mean (μ) and standard deviation (σ) of energy
energy_mean = np.mean(energy_arr)
energy_std = np.std(energy_arr)
threshold = energy_mean + energy_std

#boolean masking: True if energy > μ + σ
high_energy_mask = energy_arr > threshold

#filter popularity using boolean mask
high_energy_popularity = popularity_arr[high_energy_mask]
overall_mean_popularity = np.mean(popularity_arr)
subset_mean_popularity = np.mean(high_energy_popularity)

print(f"Energy Threshold (mean + 1 std): {threshold:.4f}")
print(f"Number of high energy tracks: {np.sum(high_energy_mask)}")
print(f"Overall Mean Popularity: {overall_mean_popularity:.4f}")
print(f"High Energy Subset Mean Popularity: {subset_mean_popularity:.4f}")

Energy Threshold (mean + 1 std): 0.8819
Number of high energy tracks: 4853
Overall Mean Popularity: 39.3298
High Energy Subset Mean Popularity: 34.0288


Comparison:The mean popularity of the high-energy subset ($34.0288$) is lower than the overall dataset average ($39.3298$).

Interpretation:This counterintuitive result shows that higher acoustic intensity or aggressiveness (tracks with energy 1 standard deviation above the norm, $> 0.8819$) does not automatically translate to mainstream appeal or higher popularity on this platform. In fact, extreme energy levels might represent niche genres (such as heavy metal, hardstyle, or underground EDM) that have a dedicated but smaller listener base.For our recommendation system, this implies that a naive model prioritizing intense features might accidentally over-recommend loud, aggressive tracks that the average listener finds jarring. The recommendation engine must carefully balance energy with other soothing or mainstream-correlated metrics (like valence or a lower danceability threshold) to maintain high user retention across broader demographics.

Stage 4 Documentation and AI Tool Reflection

I didn't use/make any functions

Key Insights:

1. Loudness Drives Energy, But Acoustic Music Slows It Down
Our analysis confirms that songs that are mixed to be physically louder naturally carry a much higher "energy" score algorithmically. Conversely, as a song introduces more traditional acoustic instruments (like a raw acoustic guitar or piano), its aggressive driving energy drops significantly. This means our system must understand that "high-energy listeners" are looking for louder, heavily produced genres (like EDM or Pop), whereas "acoustic listeners" prefer quieter, calmer musical spaces.

2. Loud and Aggressive Doesn't Mean Popular
You might think that intense, high-energy tracks dominate the charts, but the data proves otherwise. Songs with extreme energy levels (in the top 15% of the dataset) actually have a lower average popularity score (34.03) compared to the overall platform average (39.33). These highly intense songs likely belong to intense niche genres (like heavy metal or hard underground electronic music). Our recommendation engine should avoid over-recommending aggressive tracks to casual listeners, as mainstream users generally prefer more balanced or moderate energy levels.

3. Content Volume Does Not Guarantee Popularity
When analyzing our top 10 most prolific artists (those who have released the absolute highest number of tracks in this dataset), we found that a massive catalog does not correlate with track success. An artist releasing a high volume of music does not automatically mean their songs achieve a higher average popularity rating. For our recommendation strategy, this means our algorithm should prioritize actual user engagement and track quality rather than giving an unfair advantage or bias to artists simply because they have a large catalog of songs.